In [ ]:
%pip install -q shap lime

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import shap
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

feature_names = data.feature_names

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(30, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
model = MLP()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50

for epoch in range(epochs):
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    preds = model(X_test)
    preds = (preds > 0.5).float()

accuracy = accuracy_score(y_test, preds)
print("Accuracy:", accuracy)

In [ ]:
explainer = LimeTabularExplainer(
    training_data=X_train.numpy(),
    feature_names=feature_names,
    class_names=["Malignant", "Benign"],
    mode="classification"
)

def predict_fn(x):
    x = torch.tensor(x, dtype=torch.float32)
    with torch.no_grad():
        return np.concatenate([1-model(x).numpy(), model(x).numpy()], axis=1)

i = 0
exp = explainer.explain_instance(
    X_test[i].numpy(),
    predict_fn,
    num_features=10
)

exp.show_in_notebook()

In [ ]:
def shap_model(x):
    x = torch.tensor(x, dtype=torch.float32)
    with torch.no_grad():
        return model(x).detach().numpy()

In [ ]:
background = X_train[:50].numpy()

explainer_shap = shap.KernelExplainer(
    shap_model,
    background
)

shap_values = explainer_shap.shap_values(X_test[:20].numpy())

In [ ]:
shap_vals = shap_values if isinstance(shap_values, list) else shap_values

In [ ]:
shap.summary_plot(
    shap_vals,
    X_test[:20].numpy(),
    feature_names=feature_names,
    show=True
)

In [ ]:
shap.summary_plot(
    shap_vals,
    X_test[:20].numpy(),
    plot_type="bar",
    feature_names=feature_names
)